importing packages

In [31]:
import pandas as pd 
import numpy as np 
import os
from matplotlib import pyplot as plt 
import seaborn as sns 

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score

from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

import xgboost as xgboost
import pickle

In [32]:
#base directori

base_dir = "Telecommunications_Industry/csv files/"
csv_dir = "created csv"
model_dir = "saved models"


Reading data 
listing all col in tables

In [33]:
list_of_files = os.listdir(base_dir)


set_of_columns_awailable = set()

for file in list_of_files:

    if ".csv" in file:

        df = pd.read_csv(base_dir + file) 
        cols_in_df = df.columns.tolist() 

        set_of_columns_awailable.update(cols_in_df)
        print("columns in file :", file ,"are" , cols_in_df)
        print()

print("Total unique columns availabel in all files : " , len(set_of_columns_awailable))
print(set_of_columns_awailable)
        


columns in file : CustomerChurn.csv are ['LoyaltyID', 'Customer ID', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn']

columns in file : Telco_customer_churn_services.csv are ['Service ID', 'Customer ID', 'Count', 'Quarter', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long 

Note : each column have Customer id exept population table were the population count is corrensponding to zip codes 

Combining all table to form a whole dataset 

In [34]:

df = pd.read_csv(base_dir + "Telco_customer_churn.csv")

#There are two ways "Customer ID" is written in column names: one with and one without space 
#fix column name to "Customer ID" in "Telco_customer_churn.csv" file
df = df.rename(columns = {'CustomerID':'Customer ID'})

list_of_csvs = ['CustomerChurn.csv', 
                'Telco_customer_churn_demographics.csv',
                'Telco_customer_churn_location.csv',
                'Telco_customer_churn_population.csv',
                'Telco_customer_churn_services.csv',
                'Telco_customer_churn_status.csv']

for file in list_of_csvs:
    temp = pd.read_csv(base_dir + file)

    if "Customer ID" in temp.columns.tolist():
        df = pd.merge(df, temp, on = "Customer ID", how = "left", suffixes=('', '_remove'))
    else:
        df = pd.merge(df, temp, on = "Zip Code", how = "left", suffixes=('', '_remove'))
            
# remove the duplicate columns
df.drop([i for i in df.columns if 'remove' in i], axis = 1, inplace = True)

print("Total Number of columns : ", len(df.columns))
print("List of columns :", df.columns.tolist())
df.head()

Total Number of columns :  65
List of columns : ['Customer ID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason', 'LoyaltyID', 'Tenure', 'Churn', 'Age', 'Under 30', 'Married', 'Number of Dependents', 'Location ID', 'ID', 'Population', 'Service ID', 'Quarter', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Avg Monthly Long Distance Charges', 'Internet Type', 'Avg Monthly GB Download', 'Device Protection Plan', 'Premium Tech Support', 'Streaming Music', 'Unlimited Data', 'Monthly Charge', 'Total Refunds', 'Total Extra Data Charges', 'Total Lon

,Customer ID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Unlimited Data,Monthly Charge,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Status ID,Satisfaction Score,Customer Status,Churn Category
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Yes,53.85,0.0,0,20.94,129.09,SUDNGT6444,1,Churned,Competitor
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Yes,70.70,0.0,0,18.24,169.89,KZSZDV8891,2,Churned,Other
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Yes,99.65,0.0,0,97.20,917.70,EPTIUU1269,3,Churned,Other
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Yes,104.80,0.0,0,136.92,3182.97,PAJIVH8196,3,Churned,Other
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Yes,103.70,0.0,0,2172.17,7208.47,RXFOMV1173,1,Churned,Competitor


In [35]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 65 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Count                              7043 non-null   int64  
 2   Country                            7043 non-null   object 
 3   State                              7043 non-null   object 
 4   City                               7043 non-null   object 
 5   Zip Code                           7043 non-null   int64  
 6   Lat Long                           7043 non-null   object 
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Gender                             7043 non-null   object 
 10  Senior Citizen                     7043 non-null   object 
 11  Partner                            7043 non-null   objec

In [36]:
df.to_csv(csv_dir + "Telecom_Customer_Churn_Complete.csv")

#saving all atributes in one single csv file 


DATA preproccing 

Dealing with Null Values :
**Observations:** when we see the overview of dataset using df.info() we can see the datatypes of all the columns and we can also observe that all the columns have 7043 (all) not null values except the "Churn Category" and "Churn Reason" columns because they contain the reason why customer churned and how and they are only available for users that have left (churned). We can fill the null values with the constant value of "Not Churned" or "Not Applicable". - This column can be used for EDA.

In [37]:
columns_with_null_values = [(col , df[col].isnull().sum())for col in df.columns.tolist() if df[col].isnull().sum() > 0]
print("Columns with null values and their count : " , columns_with_null_values)

Columns with null values and their count :  [('Churn Reason', np.int64(5174)), ('Offer', np.int64(3877)), ('Internet Type', np.int64(1526)), ('Churn Category', np.int64(5174))]


In [38]:
df["Churn Category"].fillna("Not Applicable", inplace = True)


# replacing na values in "Churn Reason" with "Not Churned"
df["Churn Reason"].fillna("Not Churned", inplace = True)

/tmp/ipykernel_37439/3989317782.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Churn Category"].fillna("Not Applicable", inplace = True)
/tmp/ipykernel_37439/3989317782.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inpla

In [39]:
# Fill remaining null values in other columns
df["Offer"].fillna("No Offer", inplace=True)
df["Internet Type"].fillna("No Internet", inplace=True)

# Verify all null values are handled
print("Null values remaining:")
print(df.isnull().sum().sum())

Null values remaining:
0


/tmp/ipykernel_37439/3810135505.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Offer"].fillna("No Offer", inplace=True)
/tmp/ipykernel_37439/3810135505.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try us

In [40]:
for col in df.columns.tolist():
    print(col)
    print("Number of unique values:", df[col].nunique())
    print("Unique Values:", df[col].unique()[:10])
    
    if(df[col].dtype == 'int64' or df[col].dtype == 'float64'):
        print("max :", df[col].max())
        print("min :", df[col].min())

    print()

Customer ID
Number of unique values: 7043
Unique Values: ['3668-QPYBK' '9237-HQITU' '9305-CDSKC' '7892-POOKP' '0280-XJGEX'
 '4190-MFLUW' '8779-QRDMV' '1066-JKSGK' '6467-CHFZW' '8665-UTDHZ']

Count
Number of unique values: 1
Unique Values: [1]
max : 1
min : 1

Country
Number of unique values: 1
Unique Values: ['United States']

State
Number of unique values: 1
Unique Values: ['California']

City
Number of unique values: 1129
Unique Values: ['Los Angeles' 'Beverly Hills' 'Huntington Park' 'Lynwood'
 'Marina Del Rey' 'Inglewood' 'Santa Monica' 'Torrance' 'Whittier'
 'La Habra']

Zip Code
Number of unique values: 1652
Unique Values: [90003 90005 90006 90010 90015 90020 90022 90024 90028 90029]
max : 96161
min : 90001

Lat Long
Number of unique values: 1652
Unique Values: ['33.964131, -118.272783' '34.059281, -118.30742' '34.048013, -118.293953'
 '34.062125, -118.315709' '34.039224, -118.266293'
 '34.066367, -118.309868' '34.02381, -118.156582' '34.066303, -118.435479'
 '34.099869, -118.326

***replacing column values and group values in bins***

In [41]:
#columns in which we want ti replace "No internet service" with "NO"ArithmeticError

cols_to_change = ["Online Security","Online Backup", "Device Protection", "Streaming TV" , "Streaming Movies"]

#applyting that change to all the columns in list "cols_to_change"
df[cols_to_change] = np.where(df[cols_to_change] == "No internet service", "No" , df[cols_to_change])


#for "Multiple Lines" column
df["Multiple Lines"] = np.where(df["Multiple Lines"] == "No phone service", "No", df["Multiple Lines"])

cols_to_change



['Online Security',
 'Online Backup',
 'Device Protection',
 'Streaming TV',
 'Streaming Movies']

In [42]:
#group tenure in bins 
df["Tenure Bins"] = pd.cut(df['Tenure in Months'], [0, 12, 24, 48, 60, 72])
df.value_counts("Tenure Bins")

Tenure Bins
(0, 12]     2186
(24, 48]    1594
(60, 72]    1407
(12, 24]    1024
(48, 60]     832
Name: count, dtype: int64

**converting nums as strings into integerse**

In [43]:
df["Population"] = df['Population'].str.replace(',','').astype(int)

##REMOVING uneccessary columns

we can observe some columns that contain IDs represent some unique IDs correnspoding to users - let's look at then one by one and figure out if our model will be able to extract any inforamtion from those attributes or not. and there are also columns that same value for all users so there is no insformation to be learned tnited States - so state and country in our case are useless

In [44]:
#listing all columns with more than 20 unique values
for col in df.columns.tolist():
    if df[col].nunique() > 20:
        print(col, "has", df[col].nunique(), "unique values")


list_of_columns_to_drop = []

Customer ID has 7043 unique values
City has 1129 unique values
Zip Code has 1652 unique values
Lat Long has 1652 unique values
Latitude has 1652 unique values
Longitude has 1651 unique values
Tenure Months has 73 unique values
Monthly Charges has 1585 unique values
Total Charges has 6531 unique values
Churn Score has 85 unique values
CLTV has 3438 unique values
Churn Reason has 21 unique values
LoyaltyID has 7021 unique values
Tenure has 73 unique values
Age has 62 unique values
Location ID has 7043 unique values
ID has 1652 unique values
Population has 1592 unique values
Service ID has 7043 unique values
Tenure in Months has 72 unique values
Avg Monthly Long Distance Charges has 3584 unique values
Avg Monthly GB Download has 50 unique values
Monthly Charge has 1585 unique values
Total Refunds has 500 unique values
Total Long Distance Charges has 6068 unique values
Total Revenue has 6975 unique values
Status ID has 7043 unique values


#### Costemer ID

In [45]:
#Costomer ID attrbute is not useful for prediction, so we can drop it
print(df["Customer ID"].head())
print("\n Number of Unique Values:", df["Customer ID"].nunique())

0    3668-QPYBK
1    9237-HQITU
2    9305-CDSKC
3    7892-POOKP
4    0280-XJGEX
Name: Customer ID, dtype: object

 Number of Unique Values: 7043


In [46]:
#we can see that all values of that columns are have unique strings - just for identification of user
#which does not have clear interpretation and help us determine if a customer churn or not

if "Customer ID" in df.columns:
    df.set_index("Customer ID", inplace=True)
else:
    print("'Customer ID' column not found in DataFrame. Check column names or CSV loading.")

In [47]:
#couting for attribute
print(df["Count"].head())
print("\n Number of Unique Values", df["Count"].nunique())

Customer ID
3668-QPYBK    1
9237-HQITU    1
9305-CDSKC    1
7892-POOKP    1
0280-XJGEX    1
Name: Count, dtype: int64

 Number of Unique Values 1


In [ ]:
#opposite of costumer ID has the same value for each and every user regardless of whether they churn or not

if 'list_of_columns_to_drop' not in globals():
    list_of_columns_to_drop = []

list_of_columns_to_drop.append("Count")